In [ ]:
import numpy as np
import sklearn 
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from scipy.stats import norm
from scipy.optimize import minimize 
import random
import matplotlib.pyplot as plt

import time
import warnings
warnings.filterwarnings(action='ignore')

import matlab.engine
eng = matlab.engine.start_matlab()

In [ ]:
#Defining Optimizer

def GPRoptimizer(obj_func, initial_theta, bounds):
   
    def get_fun(theta): 
   #reshape X_star to fit the later input
        fun_v=obj_func(theta, eval_gradient=True)
   #print(str(fun_v))
        return fun_v
   
    funval = minimize(get_fun, initial_theta, method='L-BFGS-B', jac=True, bounds=bounds,options={'maxiter':30000})
    theta_opt=funval.x
   
    def get_fun_eval(theta):
   #reshape X_star to fit the later input
       fun_eval=obj_func(theta, eval_gradient=False)
   #print(str(theta))
   #print(str(fun_eval))
       return fun_eval
    func_min=get_fun_eval(theta_opt)
    return theta_opt, func_min

In [ ]:
#Defining Acquasition Function 

def acq_fun_EI_el(x_star,y_sample_data):
    y_prediction_mean, y_prediciton_var = gaussian_process.predict(x_star, return_std=True)#x_star[binary]
    def EI(mean, var, ff):
        Z = (ff.min() - mean) / np.sqrt(var)
        ei = (ff.min() - mean) * norm.cdf(Z) + np.sqrt(var) * norm.pdf(Z)
        return ei
    ei = EI(y_prediction_mean, y_prediciton_var, y_sample_data)
    return -ei 

In [ ]:
#Upload sample data
for batch in range(3,4):
    option1=2;
    option2=23
    sample_number=batch
    filename='training_data_sample{n}_23'.format(n=sample_number)
    f_1=open(filename+'.txt')
    data_array=np.loadtxt(f_1)
    sample_x=np.float32(data_array[:,1:data_array.shape[1]])
    sample_y=np.float32(data_array[:,0])
    
    print(np.shape(sample_x))

#Brute Search Digital Space
    qv_vector_size=int(sample_x.shape[1])  #가로축 길이 : 16
    junk_number=8 #junk_number = 8
    print(junk_number)
    x_min_junk_container=np.zeros([junk_number,qv_vector_size]) #[1,16] matrix
    ei_min_junk_container=np.zeros([junk_number]) 
    
    fom_min_junk_container = np.zeros(1)
    
#matlab 함수 이용해서 다음 FOM값 찾기
    iteration = 1000
    for i in range(1,iteration+1):
    
        print("\n iteration: ",i)
        if (i==1):
            ones_weight = []
            for a in range(int(sample_x.shape[1])):
                ones_weight.append(1)
    
            kernel = 1 * RBF(length_scale=ones_weight, length_scale_bounds=(1e-2, 1e2))
            gaussian_process = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=1,optimizer=GPRoptimizer)
            gaussian_process.fit(sample_x, sample_y)
        else:
            pass
            
        start_time = time.time()
        
        #junk가 1이 아닐 때 ei 계산
        start_time_EI = time.time()
        for idx in range(junk_number):
            with open(numpy_direc+'Lv'+str(option1)+'_W'+str(qv_vector_size)+'_cJ'+str(idx+1)+'_tJ'+str(junk_number)+'.npy', 'rb') as f:
                grid_array=np.load(f)
                current_x_junk=np.float32(grid_array) #[2^16,16], 0000..000 ~ 1111..111
                EI_pred_array=acq_fun_EI_el(current_x_junk,sample_y) #EI값 저장             
                x_initial_guess=current_x_junk[EI_pred_array.argmin(),:] #argmin : ei 최솟값 주소 [0000,,,010]
                x_min_junk_container[idx,:]=x_initial_guess # ei 최솟값 주소만을 저장 1x16
                ei_min_junk_container[idx]=EI_pred_array.min()
    
        x_initial_guess_true=x_min_junk_container[ei_min_junk_container.argmin(),:]
        
        end_time_EI = time.time()
        execution_time_EI = round((end_time_EI - start_time_EI),3)
        print(f"Execution time EI: {execution_time_EI} sec")
        
        print("x+1 : ",x_initial_guess_true)
        
        for k in range(sample_x.shape[0]):
            if(np.array_equal(x_initial_guess_true[:],sample_x[k,:])):
                random_check = "o"
                for j in range(qv_vector_size):
                    qv_random = random.random()
                    if (qv_random>1/2):
                        x_initial_guess_true[j]=1
                    else:
                        x_initial_guess_true[j]=0
            else:
                random_check = "x"
        
        print("random? :",random_check)
        sample_x=np.vstack((sample_x,x_initial_guess_true)) #sample_x에 (x+1)추가
    
        x_list = x_initial_guess_true.tolist() #array를 list로 변환, matlab engine 구동시 필요
        x_list = list(map(int, x_list))
    
        x_mat = matlab.int16(x_list)
        start_time_TMM = time.time()
        fom_min_junk_container[0]=eng.IRAR_script_weight_factor(x_mat)  #matlab으로 계산한 x+1의 fom값
        end_time_TMM = time.time()
        
        execution_time_TMM = round((end_time_TMM - start_time_TMM),3)
        print(f"TMM calculation time: {execution_time_TMM} sec")
    
        print("for x+1, fom: ", fom_min_junk_container[0])
        sample_y=np.append(sample_y,np.array([fom_min_junk_container[0]]))
    
        start_time_GP = time.time()
        gaussian_process.fit(sample_x, sample_y)
        end_time_GP = time.time()
        execution_time_GP = round((end_time_GP - start_time_GP),3)
        print(f"Execution time Gaussian: {execution_time_GP} sec")
    
    
        min_fom_x = sample_x[sample_y.argmin(),:]
        min_list = list(map(int, min_fom_x))
        result_min_x = ' '.join(map(str,min_list))
    
        print("minimum fom address: ",min_fom_x)
        print("minimum fom: ",sample_y.min())
    
    
        #save_result_txt------------------------------------------------------------------------------------------------------
        result2_str = ' '.join(map(str, x_list))
        result1_str = f"{fom_min_junk_container[0]:1.15f}"
        save_fom = '{size}_layers_{iters}_iters_BO_BF_{n}'.format(size = qv_vector_size,iters=iteration,n=sample_number)
        file_path = 'E:/JungSR/Quantum_supremacy/BO+BF/BO+BF/Result_file/'
    
        if (i==1): #initial_dataset_25
            sample_y_reshaped = sample_y.reshape(-1, 1)
            merged_array = np.hstack((sample_y_reshaped, sample_x))
            N = sample_x.shape[1]
            format_txt=' '.join(['%1.15f'] + ['%d']*N)
            np.savetxt(file_path+save_fom+'.txt', merged_array, fmt = format_txt, delimiter=' ')
            
        else:
            with open(file_path+save_fom+'.txt','a') as writing_txt:
                writing_txt.write(f"{result1_str} {result2_str} {random_check}\n")
            
        end_time = time.time()
        time_record = round(end_time - start_time,3)
        save_time = '{size}_layers_{iters}_iters_time_record_BO_BF_{n}'.format(size = qv_vector_size, iters = iteration,n=sample_number)
        with open(file_path+save_time+'.txt','a') as time_txt:
            time_txt.write(f"{i} {execution_time_EI} {execution_time_GP} {execution_time_TMM} {time_record}\n")
    
        save_min = '{size}_layers_{iters}_iters_minimum_FOM_BO_BF_{n}'.format(size = qv_vector_size, iters = iteration,n=sample_number)
        with open(file_path+save_min+".txt","a") as min_txt:
            min_txt.write(f"{i} {sample_y.min()} {result_min_x} \n")
            
        print("total time: ",time_record,'sec')
